# Task 1 — Deep Learning TinyML Optimization

This notebook starts directly from **Task 1** and is designed to run **without relying on earlier baseline cells**.

It focuses only on the **deep learning models** and applies optimization techniques **separately**:

- Pruning-like Sparsification
- Weight Clustering-like Compression
- Low-Rank Approximation
- Dynamic Quantization
- INT8 Quantization

The flow is:

1. Load the required datasets
2. Preprocess each dataset
3. Build fixed hold-out splits
4. Load the saved deep learning models
5. Apply one technique at a time
6. Evaluate each technique on each dataset
7. Plot the final comparison at the end

---

## TinyML Techniques — Reference Table

| # | Technique | a little context of what ive done | Key Reference |
|---|-----------|----------------------|---------------|
| 1 | **Pruning-like Sparsification** | Weights with small magnitude are zeroed out. The network keeps its architecture but many values become zero, enabling sparse storage and (with hardware support) faster inference. | Han, S., Pool, J., Tran, J., & Dally, W. J. (2015). *Learning both weights and connections for efficient neural networks*. NeurIPS 2015. |
| 2 | **Weight Clustering-like Compression** | All weights are snapped to a small set of shared centroid values found by k-means. Only cluster indices and centroid table need to be stored, shrinking the effective bit-width. | Han, S., Mao, H., & Dally, W. J. (2016). *Deep compression: Compressing deep neural networks with pruning, trained quantization and Huffman coding*. ICLR 2016. |
| 3 | **Low-Rank Approximation** | Dense and convolutional weight matrices are decomposed via truncated SVD and reconstructed at a lower rank, reducing parameter count while preserving the dominant directions of variation. | Denton, E., Zaremba, W., Bruna, J., LeCun, Y., & Fergus, R. (2014). *Exploiting linear structure within convolutional networks for efficient evaluation*. NeurIPS 2014. |
| 4 | **Dynamic Quantization** | Weights are quantized to INT8 ahead of time. activations are quantized on-the-fly at inference. No calibration data is required. TFLite applies this via its `DEFAULT` optimization flag. | Jacob, B., Kligys, S., Chen, B., et al. (2018). *Quantization and training of neural networks for efficient integer-arithmetic-only inference*. CVPR 2018. |
| 5 | **INT8 Static Quantization** | Both weights *and* activations are fully quantized to INT8 using a representative calibration dataset to determine per-tensor scale and zero-point. The resulting model runs entirely in integer arithmetic on supported hardware. | Jacob, B., Kligys, S., Chen, B., et al. (2018). *Quantization and training of neural networks for efficient integer-arithmetic-only inference*. CVPR 2018. |



In [1]:

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os
import json
import time
import tempfile
import copy
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import tensorflow as tf

from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.cluster import KMeans

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")


## Task 1 Configuration

In [2]:

# Dataset paths
# Update these if your local paths are different.

CICEVSE_PATH = r'C:\Users\100987869\Desktop\EV-Survey-Models2\CICEVSE2024\Subsets\CICEVSE2024_12classes_kmeans_rule_sampled1000.csv'
CICIDS_PATH  = r'c:\Users\100987869\Downloads\cic_0.01km.csv'
NFTON_PATH   = r'C:\Users\100987869\Desktop\EV-Survey-Models2\NF-Ton-IoT-V2\NF-ToN-IoT-V2.parquet'

# Saved model registry paths
MODEL_DIR   = 'models'
# Prefer the new baseline registry inside models/, but keep fallback to old root models.json.
MODELS_JSON = os.path.join(MODEL_DIR, 'models.json') if os.path.exists(os.path.join(MODEL_DIR, 'models.json')) else 'models.json'

# Deep model names expected from the saved-model registry
DEEP_MODEL_NAMES = [
    'DNN',
    'VGG16 (1D-CNN)',
    'VGG19 (1D-CNN)',
    'CNN',
    'LSTM',
]

# Task 1 hold-out split settings
TEST_SIZE = 0.20
RANDOM_STATE = 42

# NF-ToN subset size used for practical evaluation
NFTON_SUBSET_SIZE = 20000

print('Task 1 configuration loaded.')


# TinyML outputs are intentionally saved outside the baseline models/ folder.
TINYML_MODEL_ROOT = 'tinyml_deeplearning'


Task 1 configuration loaded.


In [ ]:

# ============================================================
# TinyML output path helpers
# Keeps compressed/deployment artifacts separate from baseline models.
# Output root:
#   tinyml_deeplearning/{dataset}/{technique}/{model}_{technique}.keras|tflite
# ============================================================

TINYML_MODEL_ROOT = 'tinyml_deeplearning'

_DATASET_FOLDER_MAP = {
    'CICEVSE2024': 'cicevse',
    'CICIDS2017': 'cicids',
    'NF-ToN-IoT-V2': 'nfton',
}

_TECHNIQUE_FOLDER_MAP = {
    'Pruning-like Sparsification': 'pruning',
    'Weight Clustering-like Compression': 'clustering',
    'Low-Rank Approximation': 'low_rank',
    'Dynamic Quantization': 'dynamic_quantization',
    'INT8 Quantization': 'int8_quantization',
}

_MODEL_FILE_MAP = {
    'DNN': 'dnn',
    'CNN': 'cnn',
    'VGG16 (1D-CNN)': 'vgg16',
    'VGG19 (1D-CNN)': 'vgg19',
    'LSTM': 'lstm',
}

def _safe_name(text: str) -> str:
    text = str(text).strip().lower()
    text = text.replace(' ', '_').replace('-', '_').replace('/', '_')
    text = text.replace('(', '').replace(')', '')
    while '__' in text:
        text = text.replace('__', '_')
    return text.strip('_')

def get_tinyml_dataset_folder(dataset_name: str) -> str:
    return _DATASET_FOLDER_MAP.get(dataset_name, _safe_name(dataset_name))

def get_tinyml_technique_folder(technique_name: str) -> str:
    return _TECHNIQUE_FOLDER_MAP.get(technique_name, _safe_name(technique_name))

def get_tinyml_model_base_name(model_name: str) -> str:
    return _MODEL_FILE_MAP.get(model_name, _safe_name(model_name))

def get_tinyml_save_path(dataset_name: str, model_name: str, technique_name: str, extension: str) -> str:
    dataset_folder = get_tinyml_dataset_folder(dataset_name)
    technique_folder = get_tinyml_technique_folder(technique_name)
    model_base = get_tinyml_model_base_name(model_name)

    folder = os.path.join(TINYML_MODEL_ROOT, dataset_folder, technique_folder)
    os.makedirs(folder, exist_ok=True)

    filename = f'{model_base}_{technique_folder}.{extension.lstrip(".")}'
    return os.path.join(folder, filename)

def save_tinyml_keras_model(model, dataset_name: str, model_name: str, technique_name: str) -> str:
    save_path = get_tinyml_save_path(dataset_name, model_name, technique_name, 'keras')
    model.save(save_path)
    print(f'Saved {technique_name} Keras model to: {save_path}')
    return save_path

def attach_saved_path(metrics: dict, save_path: str) -> dict:
    metrics['Saved Path'] = save_path
    return metrics


## Registry Helpers

In [3]:

def _load_registry() -> dict:
    if os.path.exists(MODELS_JSON):
        with open(MODELS_JSON, 'r') as f:
            return json.load(f)
    return {}


def _require_file(path_str: str, label: str):
    if not os.path.exists(path_str):
        raise FileNotFoundError(f'{label} not found: {path_str}')


def _normalise_path(path_str: str) -> str:
    return str(Path(path_str))


def _get_model_entry(dataset_tag: str, model_name: str):
    registry = _load_registry()

    exact_key = f'[{dataset_tag}] {model_name}'
    if exact_key in registry:
        return exact_key, registry[exact_key]

    for key, value in registry.items():
        if key.strip() == exact_key:
            return key, value

    raise KeyError(f'Model entry not found in registry: {exact_key}')


def _model_total_input_features(model) -> int:
    """Return total non-batch input features from a Keras model input shape.

    Examples:
    (None, 85) -> 85
    (None, 85, 1) -> 85
    (None, 17, 5) -> 85
    """
    input_shape = model.input_shape
    if isinstance(input_shape, list):
        input_shape = input_shape[0]

    total = 1
    for dim in input_shape[1:]:
        if dim is not None:
            total *= int(dim)
    return int(total)


def _is_feature_compatible(model, expected_features: int | None) -> bool:
    if expected_features is None:
        return True
    return _model_total_input_features(model) == int(expected_features)


def _dataset_aliases(dataset_tag: str):
    aliases = {
        'CICEVSE': ['cicevse', 'cicevse2024', 'evse'],
        'CICIDS': ['cicids', 'cicids2017', 'cic'],
        'NFToN': ['nfton', 'nf-ton', 'nf_ton', 'ton-iot', 'toniot'],
    }
    return aliases.get(dataset_tag, [dataset_tag.lower()])


def _model_aliases(model_name: str):
    lower_name = model_name.lower()
    if 'vgg16' in lower_name:
        return ['vgg16']
    if 'vgg19' in lower_name:
        return ['vgg19']
    if 'lstm' in lower_name:
        return ['lstm']
    if lower_name.strip() == 'cnn' or lower_name.strip().startswith('cnn'):
        return ['cnn']
    if 'dnn' in lower_name:
        return ['dnn']
    return [lower_name]


def _candidate_text_matches_model(text: str, model_name: str) -> bool:
    text_l = text.lower().replace(' ', '_')
    aliases = _model_aliases(model_name)

    if aliases == ['cnn']:
        # Avoid selecting VGG16/VGG19 only because their names contain 1D-CNN.
        return 'cnn' in text_l and 'vgg16' not in text_l and 'vgg19' not in text_l

    return any(alias in text_l for alias in aliases)


def _candidate_text_matches_dataset(text: str, dataset_tag: str) -> bool:
    text_l = text.lower()
    return any(alias in text_l for alias in _dataset_aliases(dataset_tag))


def _iter_candidate_model_paths(dataset_tag: str, model_name: str):
    """Yield possible .keras paths from models.json and MODEL_DIR.

    The first exact registry path is still tried, but if it is incompatible with
    the dataset feature count, the loader searches the existing saved models
    instead of deleting or overwriting anything.
    """
    registry = _load_registry()
    seen = set()

    # 1) Registry candidates, including exact key first.
    exact_key = f'[{dataset_tag}] {model_name}'
    ordered_items = []
    if exact_key in registry:
        ordered_items.append((exact_key, registry[exact_key]))
    ordered_items.extend([(k, v) for k, v in registry.items() if k != exact_key])

    for key, entry in ordered_items:
        if not isinstance(entry, dict) or 'model_path' not in entry:
            continue
        path = _normalise_path(entry['model_path'])
        if path in seen or not os.path.exists(path):
            continue
        seen.add(path)
        text = f'{key} {path}'
        if key == exact_key or _candidate_text_matches_model(text, model_name):
            yield {'source': 'models.json', 'key': key, 'path': path, 'text': text}

    # 2) Direct scan of MODEL_DIR for .keras files.
    if os.path.exists(MODEL_DIR):
        for path_obj in Path(MODEL_DIR).rglob('*.keras'):
            path = str(path_obj)
            if path in seen:
                continue
            seen.add(path)
            text = path
            if _candidate_text_matches_model(text, model_name):
                yield {'source': 'folder scan', 'key': None, 'path': path, 'text': text}


def _load_saved_keras_model(dataset_tag: str, model_name: str, expected_features: int | None = None):
    """Load a saved Keras model and verify it matches the dataset feature count.

    This protects cases like CICEVSE data with 85 features accidentally loading
    CICIDS checkpoints that expect 19 features. Existing models are not removed.
    """
    attempts = []
    compatible = []

    for cand in _iter_candidate_model_paths(dataset_tag, model_name):
        path = cand['path']
        try:
            model = tf.keras.models.load_model(path)
            actual_features = _model_total_input_features(model)
            input_shape = model.input_shape
            output_shape = model.output_shape

            row = {
                'path': path,
                'key': cand.get('key'),
                'source': cand.get('source'),
                'actual_features': actual_features,
                'input_shape': input_shape,
                'output_shape': output_shape,
                'dataset_match': _candidate_text_matches_dataset(cand['text'], dataset_tag),
                'model_match': _candidate_text_matches_model(cand['text'], model_name),
            }
            attempts.append(row)

            if expected_features is None or actual_features == int(expected_features):
                # Prefer candidates whose path/key also includes the correct dataset tag.
                score = 0
                score += 100 if actual_features == int(expected_features or actual_features) else 0
                score += 25 if row['dataset_match'] else 0
                score += 10 if cand.get('source') == 'models.json' else 0
                row['score'] = score
                compatible.append((score, model, cand, row))

        except Exception as e:
            attempts.append({
                'path': path,
                'key': cand.get('key'),
                'source': cand.get('source'),
                'error': str(e),
            })

    if compatible:
        compatible.sort(key=lambda x: x[0], reverse=True)
        score, model, cand, row = compatible[0]
        key = cand.get('key') or f'[folder scan] {dataset_tag} {model_name}'
        entry = {'model_path': cand['path'], 'loader_source': cand.get('source'), 'verified_input_features': row['actual_features']}

        print(
            f"Loaded {dataset_tag} / {model_name}: {cand['path']} | "
            f"input_shape={row['input_shape']} | features={row['actual_features']}"
        )
        return model, key, entry

    # If no compatible model exists, give a useful inventory rather than failing later in predict().
    print(f'\nNo compatible model found for {dataset_tag} / {model_name}.')
    print(f'Expected input features: {expected_features}')
    if attempts:
        print('Candidates that were found:')
        for row in attempts[:20]:
            if 'error' in row:
                print(f"  LOAD FAILED | {row['path']} | {row['error'][:120]}")
            else:
                print(f"  features={row['actual_features']} | input_shape={row['input_shape']} | path={row['path']}")
    else:
        print('No .keras candidates found in models.json or MODEL_DIR.')

    raise FileNotFoundError(
        f'No compatible saved model for {dataset_tag} / {model_name} with {expected_features} input features. '
        'The saved checkpoint may be missing or models.json may point to the wrong dataset.'
    )


## Dataset Loading and Preprocessing

In [4]:

# CICEVSE2024

_require_file(CICEVSE_PATH, 'CICEVSE2024 dataset')
df_cicevse_2024 = pd.read_csv(CICEVSE_PATH)

labelencoder_cicevse = LabelEncoder()
df_cicevse_2024.iloc[:, -1] = labelencoder_cicevse.fit_transform(df_cicevse_2024.iloc[:, -1])

if df_cicevse_2024.isnull().values.any() or np.isinf(df_cicevse_2024.select_dtypes(include=[np.number])).values.any():
    df_cicevse_2024.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_cicevse_2024.fillna(0, inplace=True)

X_cicevse = df_cicevse_2024.drop(['Label'], axis=1)
y_cicevse = df_cicevse_2024.iloc[:, -1]

print('CICEVSE2024 shape:', X_cicevse.shape)
print('CICEVSE2024 classes:', len(np.unique(y_cicevse)))


CICEVSE2024 shape: (35941, 85)
CICEVSE2024 classes: 12


In [5]:

# CICIDS2017

_require_file(CICIDS_PATH, 'CICIDS2017 dataset')
df_cicids_2017 = pd.read_csv(CICIDS_PATH)

labelencoder_cicids = LabelEncoder()
df_cicids_2017['Label'] = labelencoder_cicids.fit_transform(df_cicids_2017['Label'].astype(str))

if df_cicids_2017.isnull().values.any() or np.isinf(df_cicids_2017.select_dtypes(include=[np.number])).values.any():
    df_cicids_2017.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_cicids_2017.fillna(0, inplace=True)

X_cicids = df_cicids_2017.drop(['Label'], axis=1)
y_cicids = df_cicids_2017['Label']

print('CICIDS2017 shape:', X_cicids.shape)
print('CICIDS2017 classes:', len(np.unique(y_cicids)))


CICIDS2017 shape: (28303, 19)
CICIDS2017 classes: 2


In [6]:

# NF-ToN-IoT-V2

_require_file(NFTON_PATH, 'NF-ToN-IoT-V2 dataset')
df_nfton = pd.read_parquet(NFTON_PATH)

possible_label_cols = ['Label', 'label', 'Attack', 'attack', 'Category', 'category', 'Target', 'target', 'Description']
label_col_nfton = None

for col in possible_label_cols:
    if col in df_nfton.columns:
        label_col_nfton = col
        break

if label_col_nfton is None:
    label_col_nfton = df_nfton.columns[-1]

labelencoder_nfton = LabelEncoder()
df_nfton[label_col_nfton] = labelencoder_nfton.fit_transform(df_nfton[label_col_nfton].astype(str))

non_numeric_cols = df_nfton.drop(columns=[label_col_nfton]).select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric_cols) > 0:
    print('Dropping non-numeric NF-ToN columns:', non_numeric_cols)
    df_nfton.drop(columns=non_numeric_cols, inplace=True)

for column in df_nfton.drop(columns=[label_col_nfton]).columns:
    col_min = df_nfton[column].min()
    col_max = df_nfton[column].max()
    if col_max != col_min:
        df_nfton[column] = (df_nfton[column] - col_min) / (col_max - col_min)
    else:
        df_nfton[column] = 0

if df_nfton.isnull().values.any() or np.isinf(df_nfton.select_dtypes(include=[np.number])).values.any():
    df_nfton.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_nfton.fillna(0, inplace=True)

X_nfton = df_nfton.drop(columns=[label_col_nfton])
y_nfton = df_nfton[label_col_nfton]

if len(X_nfton) > NFTON_SUBSET_SIZE:
    X_nfton_sub, _, y_nfton_sub, _ = train_test_split(
        X_nfton,
        y_nfton,
        train_size=NFTON_SUBSET_SIZE,
        stratify=y_nfton,
        random_state=RANDOM_STATE
    )
else:
    X_nfton_sub = X_nfton.copy()
    y_nfton_sub = y_nfton.copy()

print('NF-ToN original shape:', X_nfton.shape)
print('NF-ToN subset shape:', X_nfton_sub.shape)
print('NF-ToN classes:', len(np.unique(y_nfton_sub)))


Dropping non-numeric NF-ToN columns: ['Attack']
NF-ToN original shape: (13135881, 41)
NF-ToN subset shape: (20000, 41)
NF-ToN classes: 2


## Fixed Hold-Out Splits

In [7]:

# Task 1 uses fixed hold-out splits.
# Not finished due fold-specific baseline checkpoints and test splits not being available inside this standalone notebook.

X_train_cicevse, X_test_cicevse, y_train_cicevse, y_test_cicevse = train_test_split(
    X_cicevse, y_cicevse,
    test_size=TEST_SIZE,
    stratify=y_cicevse,
    random_state=RANDOM_STATE
)

X_train_cicids, X_test_cicids, y_train_cicids, y_test_cicids = train_test_split(
    X_cicids, y_cicids,
    test_size=TEST_SIZE,
    stratify=y_cicids,
    random_state=RANDOM_STATE
)

X_train_nfton, X_test_nfton, y_train_nfton, y_test_nfton = train_test_split(
    X_nfton_sub, y_nfton_sub,
    test_size=TEST_SIZE,
    stratify=y_nfton_sub,
    random_state=RANDOM_STATE
)

print('Hold-out splits prepared.')


Hold-out splits prepared.


## Deep Model Loading

In [8]:

# CICEVSE2024 deep models
# Expected feature count comes from the actual CICEVSE test split.
expected_features_cicevse = X_test_cicevse.shape[1]
print('Expected CICEVSE2024 input features:', expected_features_cicevse)

dnn_cicevse_model, dnn_cicevse_key, dnn_cicevse_entry = _load_saved_keras_model('CICEVSE', 'DNN', expected_features=expected_features_cicevse)
vgg16_cicevse_model, vgg16_cicevse_key, vgg16_cicevse_entry = _load_saved_keras_model('CICEVSE', 'VGG16 (1D-CNN)', expected_features=expected_features_cicevse)
vgg19_cicevse_model, vgg19_cicevse_key, vgg19_cicevse_entry = _load_saved_keras_model('CICEVSE', 'VGG19 (1D-CNN)', expected_features=expected_features_cicevse)
cnn_cicevse_model, cnn_cicevse_key, cnn_cicevse_entry = _load_saved_keras_model('CICEVSE', 'CNN', expected_features=expected_features_cicevse)
lstm_cicevse_model, lstm_cicevse_key, lstm_cicevse_entry = _load_saved_keras_model('CICEVSE', 'LSTM', expected_features=expected_features_cicevse)

print('Loaded verified CICEVSE2024 deep models.')


Loaded CICEVSE2024 deep models.


In [9]:

# CICIDS2017 deep models
# Expected feature count comes from the actual CICIDS test split.
expected_features_cicids = X_test_cicids.shape[1]
print('Expected CICIDS2017 input features:', expected_features_cicids)

dnn_cicids_model, dnn_cicids_key, dnn_cicids_entry = _load_saved_keras_model('CICIDS', 'DNN', expected_features=expected_features_cicids)
vgg16_cicids_model, vgg16_cicids_key, vgg16_cicids_entry = _load_saved_keras_model('CICIDS', 'VGG16 (1D-CNN)', expected_features=expected_features_cicids)
vgg19_cicids_model, vgg19_cicids_key, vgg19_cicids_entry = _load_saved_keras_model('CICIDS', 'VGG19 (1D-CNN)', expected_features=expected_features_cicids)
cnn_cicids_model, cnn_cicids_key, cnn_cicids_entry = _load_saved_keras_model('CICIDS', 'CNN', expected_features=expected_features_cicids)
lstm_cicids_model, lstm_cicids_key, lstm_cicids_entry = _load_saved_keras_model('CICIDS', 'LSTM', expected_features=expected_features_cicids)

print('Loaded verified CICIDS2017 deep models.')


Loaded CICIDS2017 deep models.


In [10]:

# NF-ToN-IoT-V2 deep models
# Expected feature count comes from the actual NF-ToN test split.
expected_features_nfton = X_test_nfton.shape[1]
print('Expected NF-ToN-IoT-V2 input features:', expected_features_nfton)

dnn_nfton_model, dnn_nfton_key, dnn_nfton_entry = _load_saved_keras_model('NFToN', 'DNN', expected_features=expected_features_nfton)
vgg16_nfton_model, vgg16_nfton_key, vgg16_nfton_entry = _load_saved_keras_model('NFToN', 'VGG16 (1D-CNN)', expected_features=expected_features_nfton)
vgg19_nfton_model, vgg19_nfton_key, vgg19_nfton_entry = _load_saved_keras_model('NFToN', 'VGG19 (1D-CNN)', expected_features=expected_features_nfton)
cnn_nfton_model, cnn_nfton_key, cnn_nfton_entry = _load_saved_keras_model('NFToN', 'CNN', expected_features=expected_features_nfton)
lstm_nfton_model, lstm_nfton_key, lstm_nfton_entry = _load_saved_keras_model('NFToN', 'LSTM', expected_features=expected_features_nfton)

print('Loaded verified NF-ToN-IoT-V2 deep models.')


Loaded NF-ToN-IoT-V2 deep models.


In [ ]:

# Verification: each loaded model must consume the same total feature count as its dataset.

def print_model_dataset_check(dataset_name, expected_features, model_pairs):
    print(f'\n{dataset_name} model compatibility check')
    for model_name, model_obj in model_pairs:
        total_features = _model_total_input_features(model_obj)
        status = 'OK' if total_features == expected_features else 'MISMATCH'
        print(f'{model_name:16s} | input_shape={str(model_obj.input_shape):18s} | total_features={total_features:4d} | expected={expected_features:4d} | {status}')

print_model_dataset_check('CICEVSE2024', expected_features_cicevse, [
    ('DNN', dnn_cicevse_model),
    ('VGG16', vgg16_cicevse_model),
    ('VGG19', vgg19_cicevse_model),
    ('CNN', cnn_cicevse_model),
    ('LSTM', lstm_cicevse_model),
])

print_model_dataset_check('CICIDS2017', expected_features_cicids, [
    ('DNN', dnn_cicids_model),
    ('VGG16', vgg16_cicids_model),
    ('VGG19', vgg19_cicids_model),
    ('CNN', cnn_cicids_model),
    ('LSTM', lstm_cicids_model),
])

print_model_dataset_check('NF-ToN-IoT-V2', expected_features_nfton, [
    ('DNN', dnn_nfton_model),
    ('VGG16', vgg16_nfton_model),
    ('VGG19', vgg19_nfton_model),
    ('CNN', cnn_nfton_model),
    ('LSTM', lstm_nfton_model),
])


## TinyML Helpers

In [11]:

def get_model_family(model_name: str) -> str:
    return 'Deep Learning'


def compute_macro_f1(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    f1_vals = []
    for cls_name, cls_metrics in report.items():
        if cls_name in ['accuracy', 'macro avg', 'weighted avg']:
            continue
        if isinstance(cls_metrics, dict) and 'f1-score' in cls_metrics:
            f1_vals.append(cls_metrics['f1-score'])
    return float(np.mean(f1_vals)) if len(f1_vals) > 0 else np.nan


def get_model_input_type(model_name: str) -> str:
    lower_name = model_name.lower()
    if 'dnn' in lower_name:
        return 'dnn'
    elif 'lstm' in lower_name:
        return 'lstm'
    else:
        return 'cnn_like'


def prepare_input_for_model(X_df, model_name: str):
    input_type = get_model_input_type(model_name)

    if input_type == 'dnn':
        return X_df.values.astype(np.float32)

    elif input_type == 'cnn_like':
        return X_df.values.astype(np.float32).reshape(-1, X_df.shape[1], 1)

    elif input_type == 'lstm':
        arr = X_df.values.astype(np.float32)
        chunk_size = 5
        remainder = arr.shape[1] % chunk_size
        if remainder != 0:
            pad_width = chunk_size - remainder
            arr = np.concatenate([arr, np.zeros((arr.shape[0], pad_width), dtype=np.float32)], axis=1)
        timesteps = arr.shape[1] // chunk_size
        return arr.reshape((arr.shape[0], timesteps, chunk_size))

    else:
        raise ValueError(f'Unsupported input type for model: {model_name}')


def evaluate_keras_model(model, X_test_ready, y_test, model_name: str, model_size_bytes=None):
    start = time.time()
    y_prob = model.predict(X_test_ready, verbose=0)
    infer_time_s = time.time() - start

    y_pred = np.argmax(y_prob, axis=1)
    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)

    return {
        'Model': model_name,
        'Accuracy (%)': accuracy_score(y_test, y_pred) * 100,
        'Precision (%)': p * 100,
        'Recall (%)': r * 100,
        'Weighted F1 (%)': f * 100,
        'Macro F1 (%)': compute_macro_f1(y_test, y_pred) * 100,
        'Infer. Time (ms/sample)': (infer_time_s / len(X_test_ready)) * 1000,
        'Model Size (MB)': (model_size_bytes / (1024 * 1024)) if model_size_bytes is not None else np.nan,
    }


def get_keras_model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix='.keras', delete=False) as tmp:
        temp_path = tmp.name
    model.save(temp_path)
    size_mb = os.path.getsize(temp_path) / (1024 * 1024)
    os.remove(temp_path)
    return size_mb


def get_file_size_mb(path_str: str):
    return os.path.getsize(path_str) / (1024 * 1024)


def clone_model_with_weights(model):
    cloned = tf.keras.models.clone_model(model)
    cloned.set_weights(model.get_weights())
    return cloned


In [12]:

def apply_pruning_like(model, prune_ratio=0.20):
    pruned_model = clone_model_with_weights(model)

    pruned_weights = []
    for w in pruned_model.get_weights():
        if w.ndim >= 2:
            threshold = np.quantile(np.abs(w), prune_ratio)
            w = np.where(np.abs(w) < threshold, 0.0, w)
        pruned_weights.append(w)

    pruned_model.set_weights(pruned_weights)
    return pruned_model


def apply_weight_clustering_like(model, n_clusters=16):
    clustered_model = clone_model_with_weights(model)
    new_weights = []

    for w in clustered_model.get_weights():
        if w.ndim >= 2 and np.prod(w.shape) >= n_clusters:
            flat_w = w.reshape(-1, 1)
            kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
            cluster_ids = kmeans.fit_predict(flat_w)
            cluster_centers = kmeans.cluster_centers_.flatten()
            flat_new = cluster_centers[cluster_ids]
            w = flat_new.reshape(w.shape)
        new_weights.append(w)

    clustered_model.set_weights(new_weights)
    return clustered_model


def apply_low_rank_approximation(model, rank_ratio=0.50):
    reduced_model = clone_model_with_weights(model)
    new_weights = []

    for w in reduced_model.get_weights():
        if w.ndim == 2:
            u, s, vt = np.linalg.svd(w, full_matrices=False)
            rank_k = max(1, int(min(w.shape) * rank_ratio))
            w = (u[:, :rank_k] @ np.diag(s[:rank_k]) @ vt[:rank_k, :]).astype(np.float32)

        elif w.ndim == 3:
            original_shape = w.shape
            flat_w = w.reshape(original_shape[0] * original_shape[1], original_shape[2])
            u, s, vt = np.linalg.svd(flat_w, full_matrices=False)
            rank_k = max(1, int(min(flat_w.shape) * rank_ratio))
            flat_w = (u[:, :rank_k] @ np.diag(s[:rank_k]) @ vt[:rank_k, :]).astype(np.float32)
            w = flat_w.reshape(original_shape)

        new_weights.append(w)

    reduced_model.set_weights(new_weights)
    return reduced_model


In [13]:
def representative_dataset_generator(X_ready, n_samples=200):
    X_small = X_ready[:min(n_samples, len(X_ready))]
    for i in range(len(X_small)):
        yield [X_small[i:i+1].astype(np.float32)]


def export_tflite_model(model, output_path, quantization='dynamic', representative_data=None):
    """Convert a Keras model to TFLite.

    quantization : 'dynamic'  — weight-only INT8, activations quantized at runtime
                   'int8'     — full INT8 static quantization (requires representative_data)
    """
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    if quantization == 'dynamic':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    elif quantization == 'int8':
        if representative_data is None:
            raise ValueError("representative_data must be provided for int8 quantization")
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_data
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type  = tf.int8
        converter.inference_output_type = tf.int8

    else:
        raise ValueError("quantization must be 'dynamic' or 'int8'")

    tflite_model = converter.convert()
    with open(output_path, 'wb') as f:
        f.write(tflite_model)


def _cast_input_for_interpreter(x_float32, input_detail):
    """Scale and cast a float32 sample to match the interpreter's expected input type.

    For INT8 models the converter stores a (scale, zero_point) pair in the
    input_detail quantization tuple.  We apply the inverse quantization mapping
    so the integer values the model expects are correct.
    """
    dtype = input_detail['dtype']
    if dtype == np.float32:
        return x_float32

    # INT8 / UINT8 path
    scale, zero_point = input_detail['quantization']
    if scale == 0:          # safety fallback — should not happen on a properly calibrated model
        scale = 1.0
    x_quant = (x_float32 / scale) + zero_point
    return np.clip(np.round(x_quant), np.iinfo(dtype).min, np.iinfo(dtype).max).astype(dtype)


def _dequantize_output(raw_output, output_detail):
    """Convert integer output back to float32 probabilities when needed."""
    dtype = output_detail['dtype']
    if dtype == np.float32:
        return raw_output.astype(np.float32)
    scale, zero_point = output_detail['quantization']
    if scale == 0:
        scale = 1.0
    return ((raw_output.astype(np.float32) - zero_point) * scale)


def evaluate_tflite_model(tflite_path, X_test_ready, y_test, model_name: str):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    preds = []
    start = time.time()

    for i in range(len(X_test_ready)):
        x_f32 = X_test_ready[i:i+1].astype(np.float32)
        x_in  = _cast_input_for_interpreter(x_f32, input_details[0])
        interpreter.set_tensor(input_details[0]['index'], x_in)
        interpreter.invoke()
        raw_out = interpreter.get_tensor(output_details[0]['index'])
        out_f32 = _dequantize_output(raw_out, output_details[0])
        preds.append(np.argmax(out_f32, axis=1)[0])

    infer_time_s = time.time() - start
    y_pred = np.array(preds)

    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)

    return {
        'Model':                          model_name,
        'Accuracy (%)':                   accuracy_score(y_test, y_pred) * 100,
        'Precision (%)':                  p * 100,
        'Recall (%)':                     r * 100,
        'Weighted F1 (%)':                f * 100,
        'Macro F1 (%)':                   compute_macro_f1(y_test, y_pred) * 100,
        'Infer. Time (ms/sample)':        (infer_time_s / len(X_test_ready)) * 1000,
        'Model Size (MB)':                get_file_size_mb(tflite_path),
    }


In [14]:

def make_result_row(dataset_name, technique_name, baseline_model_name, tiny_model_name, baseline_metrics, tiny_metrics, note=''):
    return {
        'Dataset': dataset_name,
        'Technique': technique_name,
        'Baseline Model': baseline_model_name,
        'Tiny Model': tiny_model_name,
        'Baseline Accuracy (%)': baseline_metrics['Accuracy (%)'],
        'Tiny Accuracy (%)': tiny_metrics['Accuracy (%)'],
        'Baseline Precision (%)': baseline_metrics['Precision (%)'],
        'Tiny Precision (%)': tiny_metrics['Precision (%)'],
        'Baseline Recall (%)': baseline_metrics['Recall (%)'],
        'Tiny Recall (%)': tiny_metrics['Recall (%)'],
        'Baseline Weighted F1 (%)': baseline_metrics['Weighted F1 (%)'],
        'Tiny Weighted F1 (%)': tiny_metrics['Weighted F1 (%)'],
        'Baseline Macro F1 (%)': baseline_metrics['Macro F1 (%)'],
        'Tiny Macro F1 (%)': tiny_metrics['Macro F1 (%)'],
        'Baseline Infer. Time (ms/sample)': baseline_metrics['Infer. Time (ms/sample)'],
        'Tiny Infer. Time (ms/sample)': tiny_metrics['Infer. Time (ms/sample)'],
        'Baseline Model Size (MB)': baseline_metrics['Model Size (MB)'],
        'Tiny Model Size (MB)': tiny_metrics['Model Size (MB)'],
        'Weighted F1 Change (%)': tiny_metrics['Weighted F1 (%)'] - baseline_metrics['Weighted F1 (%)'],
        'Macro F1 Change (%)': tiny_metrics['Macro F1 (%)'] - baseline_metrics['Macro F1 (%)'],
        'Infer. Time Change (%)': ((tiny_metrics['Infer. Time (ms/sample)'] - baseline_metrics['Infer. Time (ms/sample)']) / max(baseline_metrics['Infer. Time (ms/sample)'], 1e-9)) * 100,
        'Model Size Change (%)': ((tiny_metrics['Model Size (MB)'] - baseline_metrics['Model Size (MB)']) / max(baseline_metrics['Model Size (MB)'], 1e-9)) * 100,
        'Saved Path': tiny_metrics.get('Saved Path', ''),
        'Note': note,
    }


def plot_baseline_vs_tiny(result_df, dataset_name, technique_name, metric_col_baseline, metric_col_tiny, xlabel_text, title_suffix):
    plot_df = result_df[(result_df['Dataset'] == dataset_name) & (result_df['Technique'] == technique_name)].copy()

    labels = plot_df['Tiny Model'].tolist()
    baseline_vals = plot_df[metric_col_baseline].tolist()
    tiny_vals = plot_df[metric_col_tiny].tolist()

    y = np.arange(len(labels))
    height = 0.35

    fig, ax = plt.subplots(figsize=(12, max(5, len(labels) * 0.55)))
    bars1 = ax.barh(y - height / 2, baseline_vals, height, label='Baseline', color=sns.color_palette('muted')[0])
    bars2 = ax.barh(y + height / 2, tiny_vals, height, label='TinyML', color=sns.color_palette('muted')[1])

    for bar, val in zip(bars1, baseline_vals):
        ax.text(val + 0.2, bar.get_y() + bar.get_height() / 2, f'{val:.2f}', va='center', fontsize=8)

    for bar, val in zip(bars2, tiny_vals):
        ax.text(val + 0.2, bar.get_y() + bar.get_height() / 2, f'{val:.2f}', va='center', fontsize=8)

    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel(xlabel_text)
    ax.set_ylabel('Model')
    ax.set_title(f'{dataset_name} — {technique_name} — {title_suffix}')
    ax.grid(True, axis='x', linestyle='--', alpha=0.5)
    ax.legend()
    plt.tight_layout()
    plt.show()


## Baseline Metrics for the Deep Models

In [15]:

# Baseline metrics for CICEVSE2024 deep models

baseline_rows_cicevse = []

for model_name, model_obj in [
    ('DNN', dnn_cicevse_model),
    ('VGG16 (1D-CNN)', vgg16_cicevse_model),
    ('VGG19 (1D-CNN)', vgg19_cicevse_model),
    ('CNN', cnn_cicevse_model),
    ('LSTM', lstm_cicevse_model),
  

]:
    X_test_ready = prepare_input_for_model(X_test_cicevse, model_name)
    model_size_mb = get_keras_model_size_mb(model_obj) * 1024 * 1024
    metrics = evaluate_keras_model(model_obj, X_test_ready, y_test_cicevse, model_name, model_size_bytes=model_size_mb)
    baseline_rows_cicevse.append(metrics)

baseline_df_cicevse = pd.DataFrame(baseline_rows_cicevse)
display(baseline_df_cicevse)


,Model,Accuracy (%),Precision (%),Recall (%),Weighted F1 (%),Macro F1 (%),Infer. Time (ms/sample),Model Size (MB)
0,DNN,51.397969,71.285165,51.397969,45.992075,45.651833,0.064305,16.583460
1,VGG16 (1D-CNN),59.465851,65.214597,59.465851,53.847212,54.189294,0.406534,74.435253
2,VGG19 (1D-CNN),21.407706,4.582899,21.407706,7.549601,2.938818,0.522686,94.715025
3,CNN,57.087217,57.219110,57.087217,53.934688,53.258338,0.163328,5.858548
4,LSTM,60.509111,58.647871,60.509111,53.180515,54.522156,0.065363,0.382554


In [16]:

# Baseline metrics for CICIDS2017 deep models

baseline_rows_cicids = []

for model_name, model_obj in [
    ('DNN', dnn_cicids_model), 
    ('VGG16 (1D-CNN)', vgg16_cicids_model),
    ('VGG19 (1D-CNN)', vgg19_cicids_model),
    ('CNN', cnn_cicids_model),
    ('LSTM', lstm_cicids_model),
    
]:
    X_test_ready = prepare_input_for_model(X_test_cicids, model_name)
    model_size_mb = get_keras_model_size_mb(model_obj) * 1024 * 1024
    metrics = evaluate_keras_model(model_obj, X_test_ready, y_test_cicids, model_name, model_size_bytes=model_size_mb)
    baseline_rows_cicids.append(metrics)

baseline_df_cicids = pd.DataFrame(baseline_rows_cicids)
display(baseline_df_cicids)


,Model,Accuracy (%),Precision (%),Recall (%),Weighted F1 (%),Macro F1 (%),Infer. Time (ms/sample),Model Size (MB)
0,DNN,57.586999,62.488000,57.586999,59.856317,41.146806,0.075177,15.795259
1,VGG16 (1D-CNN),53.011835,61.291820,53.011835,56.633616,39.215307,0.309818,74.315339
2,VGG19 (1D-CNN),64.423247,75.045795,64.423247,67.758837,57.084321,0.418013,94.597700
3,CNN,80.003533,64.107480,80.003533,71.178806,44.445535,0.120391,5.843764
4,LSTM,80.021198,64.110302,80.021198,71.187536,44.450986,0.055777,0.375109


In [17]:

# Baseline metrics for NF-ToN-IoT-V2 deep models

baseline_rows_nfton = []

for model_name, model_obj in [
    ('DNN', dnn_nfton_model),
    ('VGG16 (1D-CNN)', vgg16_nfton_model),
    ('VGG19 (1D-CNN)', vgg19_nfton_model),
    ('CNN', cnn_nfton_model),
    ('LSTM', lstm_nfton_model),
    
]:
    X_test_ready = prepare_input_for_model(X_test_nfton, model_name)
    model_size_mb = get_keras_model_size_mb(model_obj) * 1024 * 1024
    metrics = evaluate_keras_model(model_obj, X_test_ready, y_test_nfton, model_name, model_size_bytes=model_size_mb)
    baseline_rows_nfton.append(metrics)

baseline_df_nfton = pd.DataFrame(baseline_rows_nfton)
display(baseline_df_nfton)


,Model,Accuracy (%),Precision (%),Recall (%),Weighted F1 (%),Macro F1 (%),Infer. Time (ms/sample),Model Size (MB)
0,DNN,97.450,97.454858,97.450,97.452158,96.802508,0.077564,16.053071
1,VGG16 (1D-CNN),96.025,96.152499,96.025,95.938445,94.797440,0.298573,74.317944
2,VGG19 (1D-CNN),72.575,52.671306,72.575,61.041641,42.054179,0.380053,94.597709
3,CNN,97.575,97.578904,97.575,97.556702,96.908749,0.105730,5.843784
4,LSTM,95.550,95.689190,95.550,95.444359,94.155255,0.059130,0.375109


## Pruning-like Sparsification

In [18]:

# Pruning-like Sparsification — CICEVSE2024

pruning_rows_cicevse2024 = []

for model_name, model_obj in [
    ('DNN', dnn_cicevse_model),
    ('VGG16 (1D-CNN)', vgg16_cicevse_model),
    ('VGG19 (1D-CNN)', vgg19_cicevse_model),
    ('CNN', cnn_cicevse_model),
    ('LSTM', lstm_cicevse_model),
    
]:
    X_test_ready = prepare_input_for_model(X_test_cicevse, model_name)

    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()

    pruned_model = apply_pruning_like(model_obj, prune_ratio=0.20)
    save_path = save_tinyml_keras_model(pruned_model, 'CICEVSE2024', model_name, 'Pruning-like Sparsification')
    pruned_size_mb = get_keras_model_size_mb(pruned_model) * 1024 * 1024
    tiny_metrics = evaluate_keras_model(pruned_model, X_test_ready, y_test_cicevse, f'{model_name} [Pruned]', model_size_bytes=pruned_size_mb)
    tiny_metrics['Saved Path'] = save_path

    pruning_rows_cicevse2024.append(
        make_result_row(
            'CICEVSE2024',
            'Pruning-like Sparsification',
            model_name,
            f'{model_name} [Pruned]',
            baseline_metrics,
            tiny_metrics,
            note='Weights below the selected quantile threshold were zeroed after loading the saved model.'
        )
    )

pruning_df_cicevse2024 = pd.DataFrame(pruning_rows_cicevse2024)
display(pruning_df_cicevse2024)

,Dataset,Technique,Baseline Model,Tiny Model,Baseline Accuracy (%),Tiny Accuracy (%),Baseline Precision (%),Tiny Precision (%),Baseline Recall (%),Tiny Recall (%),...,Tiny Macro F1 (%),Baseline Infer. Time (ms/sample),Tiny Infer. Time (ms/sample),Baseline Model Size (MB),Tiny Model Size (MB),Weighted F1 Change (%),Macro F1 Change (%),Infer. Time Change (%),Model Size Change (%),Note
0,CICEVSE2024,Pruning-like Sparsification,DNN,DNN [Pruned],51.397969,48.212547,71.285165,67.882012,51.397969,48.212547,...,42.670577,0.064305,0.074317,16.583460,16.583460,-0.710380,-2.981256,15.568526,0.0,Weights below the selected quantile threshold ...
1,CICEVSE2024,Pruning-like Sparsification,VGG16 (1D-CNN),VGG16 (1D-CNN) [Pruned],59.465851,60.272639,65.214597,65.552568,59.465851,60.272639,...,55.064930,0.406534,0.386964,74.435253,74.435253,0.407562,0.875637,-4.813805,0.0,Weights below the selected quantile threshold ...
2,CICEVSE2024,Pruning-like Sparsification,VGG19 (1D-CNN),VGG19 (1D-CNN) [Pruned],21.407706,21.407706,4.582899,4.582899,21.407706,21.407706,...,2.938818,0.522686,0.487363,94.715025,94.715025,0.000000,0.000000,-6.757883,0.0,Weights below the selected quantile threshold ...
3,CICEVSE2024,Pruning-like Sparsification,CNN,CNN [Pruned],57.087217,57.115037,57.219110,57.210702,57.087217,57.115037,...,54.030128,0.163328,0.136988,5.858548,5.858548,-0.025630,0.771790,-16.126939,0.0,Weights below the selected quantile threshold ...
4,CICEVSE2024,Pruning-like Sparsification,LSTM,LSTM [Pruned],60.509111,60.453471,58.647871,58.656240,60.509111,60.453471,...,54.184280,0.065363,0.064755,0.382554,0.382554,0.930840,-0.337876,-0.929487,0.0,Weights below the selected quantile threshold ...


In [19]:
# Pruning-like Sparsification — CICIDS2017

pruning_rows_cicids2017 = []

for model_name, model_obj in [
    ('DNN',            dnn_cicids_model),
    ('VGG16 (1D-CNN)', vgg16_cicids_model),
    ('VGG19 (1D-CNN)', vgg19_cicids_model),
    ('CNN',            cnn_cicids_model),
    ('LSTM',           lstm_cicids_model),
]:
    X_test_ready = prepare_input_for_model(X_test_cicids, model_name)

    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()

    pruned_model    = apply_pruning_like(model_obj, prune_ratio=0.20)
    pruned_size_mb  = get_keras_model_size_mb(pruned_model) * 1024 * 1024
    tiny_metrics    = evaluate_keras_model(pruned_model, X_test_ready, y_test_cicids,
                                           f'{model_name} [Pruned]', model_size_bytes=pruned_size_mb)

    pruning_rows_cicids2017.append(
        make_result_row(
            'CICIDS2017',
            'Pruning-like Sparsification',
            model_name,
            f'{model_name} [Pruned]',
            baseline_metrics,
            tiny_metrics,
            note='Weights below the selected quantile threshold were zeroed after loading the saved model.'
        )
    )

pruning_df_cicids2017 = pd.DataFrame(pruning_rows_cicids2017)
display(pruning_df_cicids2017)


,Dataset,Technique,Baseline Model,Tiny Model,Baseline Accuracy (%),Tiny Accuracy (%),Baseline Precision (%),Tiny Precision (%),Baseline Recall (%),Tiny Recall (%),...,Tiny Macro F1 (%),Baseline Infer. Time (ms/sample),Tiny Infer. Time (ms/sample),Baseline Model Size (MB),Tiny Model Size (MB),Weighted F1 Change (%),Macro F1 Change (%),Infer. Time Change (%),Model Size Change (%),Note
0,CICIDS2017,Pruning-like Sparsification,DNN,DNN [Pruned],57.586999,57.551669,62.488000,62.455634,57.586999,57.551669,...,41.097782,0.075177,0.064054,15.795259,15.795259,-0.033439,-0.049024,-14.794912,0.0,Weights below the selected quantile threshold ...
1,CICIDS2017,Pruning-like Sparsification,VGG16 (1D-CNN),VGG16 (1D-CNN) [Pruned],53.011835,52.888182,61.291820,61.074219,53.011835,52.888182,...,38.908730,0.309818,0.285025,74.315339,74.315339,-0.147325,-0.306578,-8.002217,0.0,Weights below the selected quantile threshold ...
2,CICIDS2017,Pruning-like Sparsification,VGG19 (1D-CNN),VGG19 (1D-CNN) [Pruned],64.423247,64.529235,75.045795,75.060612,64.423247,64.529235,...,57.151658,0.418013,0.374345,94.597700,94.597700,0.087010,0.067337,-10.446551,0.0,Weights below the selected quantile threshold ...
3,CICIDS2017,Pruning-like Sparsification,CNN,CNN [Pruned],80.003533,80.074192,64.107480,64.118762,80.003533,80.074192,...,44.467334,0.120391,0.098652,5.843764,5.843764,0.034911,0.021799,-18.056990,0.0,Weights below the selected quantile threshold ...
4,CICIDS2017,Pruning-like Sparsification,LSTM,LSTM [Pruned],80.021198,80.021198,64.110302,64.110302,80.021198,80.021198,...,44.450986,0.055777,0.049244,0.375109,0.375109,0.000000,0.000000,-11.711059,0.0,Weights below the selected quantile threshold ...


In [20]:

# Pruning-like Sparsification — NF-ToN-IoT-V2

pruning_rows_nftoniotv2 = []

for model_name, model_obj in [
    ('DNN', dnn_nfton_model),
    ('VGG16 (1D-CNN)', vgg16_nfton_model),
    ('VGG19 (1D-CNN)', vgg19_nfton_model),
    ('CNN', cnn_nfton_model),
    ('LSTM', lstm_nfton_model)
]:
    X_test_ready = prepare_input_for_model(X_test_nfton, model_name)

    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()

    pruned_model = apply_pruning_like(model_obj, prune_ratio=0.20)
    save_path = save_tinyml_keras_model(pruned_model, 'NF-ToN-IoT-V2', model_name, 'Pruning-like Sparsification')
    pruned_size_mb = get_keras_model_size_mb(pruned_model) * 1024 * 1024
    tiny_metrics = evaluate_keras_model(pruned_model, X_test_ready, y_test_nfton, f'{model_name} [Pruned]', model_size_bytes=pruned_size_mb)
    tiny_metrics['Saved Path'] = save_path

    pruning_rows_nftoniotv2.append(
        make_result_row(
            'NF-ToN-IoT-V2',
            'Pruning-like Sparsification',
            model_name,
            f'{model_name} [Pruned]',
            baseline_metrics,
            tiny_metrics,
            #note='Weights below the selected quantile threshold were zeroed after loading the saved model.'
        )
    )

pruning_df_nftoniotv2 = pd.DataFrame(pruning_rows_nftoniotv2)
display(pruning_df_nftoniotv2)

,Dataset,Technique,Baseline Model,Tiny Model,Baseline Accuracy (%),Tiny Accuracy (%),Baseline Precision (%),Tiny Precision (%),Baseline Recall (%),Tiny Recall (%),...,Tiny Macro F1 (%),Baseline Infer. Time (ms/sample),Tiny Infer. Time (ms/sample),Baseline Model Size (MB),Tiny Model Size (MB),Weighted F1 Change (%),Macro F1 Change (%),Infer. Time Change (%),Model Size Change (%),Note
0,NF-ToN-IoT-V2,Pruning-like Sparsification,DNN,DNN [Pruned],97.450,97.725,97.454858,97.719684,97.450,97.725,...,97.133522,0.077564,0.056292,16.053071,16.053071,0.269261,0.331014,-27.425566,0.0,
1,NF-ToN-IoT-V2,Pruning-like Sparsification,VGG16 (1D-CNN),VGG16 (1D-CNN) [Pruned],96.025,95.875,96.152499,96.021264,96.025,95.875,...,94.587324,0.298573,0.305757,74.317944,74.317944,-0.159147,-0.210116,2.406142,0.0,
2,NF-ToN-IoT-V2,Pruning-like Sparsification,VGG19 (1D-CNN),VGG19 (1D-CNN) [Pruned],72.575,72.575,52.671306,52.671306,72.575,72.575,...,42.054179,0.380053,0.404121,94.597709,94.597709,0.000000,0.000000,6.332804,0.0,
3,NF-ToN-IoT-V2,Pruning-like Sparsification,CNN,CNN [Pruned],97.575,97.700,97.578904,97.703298,97.575,97.700,...,97.070690,0.105730,0.117309,5.843784,5.843784,0.127005,0.161941,10.951559,0.0,
4,NF-ToN-IoT-V2,Pruning-like Sparsification,LSTM,LSTM [Pruned],95.550,95.500,95.689190,95.629882,95.550,95.500,...,94.093372,0.059130,0.054372,0.375109,0.375109,-0.049568,-0.061883,-8.047019,0.0,


## Weight Clustering-like Compression

In [21]:

# Weight Clustering-like Compression — CICEVSE2024

cluster_rows_cicevse2024 = []

for model_name, model_obj in [
    ('DNN', dnn_cicevse_model),
    ('VGG16 (1D-CNN)', vgg16_cicevse_model),
    ('VGG19 (1D-CNN)', vgg19_cicevse_model),
    ('CNN', cnn_cicevse_model),
    ('LSTM', lstm_cicevse_model),
    
]:
    X_test_ready = prepare_input_for_model(X_test_cicevse, model_name)

    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()

    clustered_model = apply_weight_clustering_like(model_obj, n_clusters=16)
    save_path = save_tinyml_keras_model(clustered_model, 'CICEVSE2024', model_name, 'Weight Clustering-like Compression')
    clustered_size_mb = get_keras_model_size_mb(clustered_model) * 1024 * 1024
    tiny_metrics = evaluate_keras_model(clustered_model, X_test_ready, y_test_cicevse, f'{model_name} [Clustered]', model_size_bytes=clustered_size_mb)
    tiny_metrics['Saved Path'] = save_path

    cluster_rows_cicevse2024.append(
        make_result_row(
            'CICEVSE2024',
            'Weight Clustering-like Compression',
            model_name,
            f'{model_name} [Clustered]',
            baseline_metrics,
            tiny_metrics,
            note='Weight values were reassigned to learned shared cluster centers.'
        )
    )

cluster_df_cicevse2024 = pd.DataFrame(cluster_rows_cicevse2024)
display(cluster_df_cicevse2024)

,Dataset,Technique,Baseline Model,Tiny Model,Baseline Accuracy (%),Tiny Accuracy (%),Baseline Precision (%),Tiny Precision (%),Baseline Recall (%),Tiny Recall (%),...,Tiny Macro F1 (%),Baseline Infer. Time (ms/sample),Tiny Infer. Time (ms/sample),Baseline Model Size (MB),Tiny Model Size (MB),Weighted F1 Change (%),Macro F1 Change (%),Infer. Time Change (%),Model Size Change (%),Note
0,CICEVSE2024,Weight Clustering-like Compression,DNN,DNN [Clustered],51.397969,51.231047,71.285165,63.383511,51.397969,51.231047,...,47.027837,0.064305,0.068443,16.583460,16.583460,-0.580787,1.376005,6.434436,0.0,Weight values were reassigned to learned share...
1,CICEVSE2024,Weight Clustering-like Compression,VGG16 (1D-CNN),VGG16 (1D-CNN) [Clustered],59.465851,59.076367,65.214597,63.543923,59.465851,59.076367,...,49.321236,0.406534,0.388266,74.435253,74.435253,-0.800867,-4.868057,-4.493610,0.0,Weight values were reassigned to learned share...
2,CICEVSE2024,Weight Clustering-like Compression,VGG19 (1D-CNN),VGG19 (1D-CNN) [Clustered],21.407706,21.407706,4.582899,4.582899,21.407706,21.407706,...,2.938818,0.522686,0.462937,94.715025,94.715025,0.000000,0.000000,-11.431049,0.0,Weight values were reassigned to learned share...
3,CICEVSE2024,Weight Clustering-like Compression,CNN,CNN [Clustered],57.087217,57.198498,57.219110,59.842094,57.087217,57.198498,...,53.260602,0.163328,0.141886,5.858548,5.858548,0.242339,0.002264,-13.127932,0.0,Weight values were reassigned to learned share...
4,CICEVSE2024,Weight Clustering-like Compression,LSTM,LSTM [Clustered],60.509111,60.036166,58.647871,61.200907,60.509111,60.036166,...,52.523221,0.065363,0.065582,0.382554,0.382554,-1.704861,-1.998935,0.335232,0.0,Weight values were reassigned to learned share...


In [22]:

# Weight Clustering-like Compression — CICIDS2017

cluster_rows_cicids2017 = []

for model_name, model_obj in [
    ('DNN', dnn_cicids_model),
    ('VGG16 (1D-CNN)', vgg16_cicids_model),
    ('VGG19 (1D-CNN)', vgg19_cicids_model),
    ('CNN', cnn_cicids_model),
    ('LSTM', lstm_cicids_model),
   
]:
    X_test_ready = prepare_input_for_model(X_test_cicids, model_name)
    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()

    clustered_model = apply_weight_clustering_like(model_obj, n_clusters=16)
    save_path = save_tinyml_keras_model(clustered_model, 'CICIDS2017', model_name, 'Weight Clustering-like Compression')
    clustered_size_mb = get_keras_model_size_mb(clustered_model) * 1024 * 1024
    tiny_metrics = evaluate_keras_model(clustered_model, X_test_ready, y_test_cicids, f'{model_name} [Clustered]', model_size_bytes=clustered_size_mb)
    tiny_metrics['Saved Path'] = save_path

    cluster_rows_cicids2017.append(
        make_result_row(
            'CICIDS2017',
            'Weight Clustering-like Compression',
            model_name,
            f'{model_name} [Clustered]',
            baseline_metrics,
            tiny_metrics,
            note='Weight values were reassigned to learned shared cluster centers.'
        )
    )

cluster_df_cicids2017 = pd.DataFrame(cluster_rows_cicids2017)
display(cluster_df_cicids2017)

,Dataset,Technique,Baseline Model,Tiny Model,Baseline Accuracy (%),Tiny Accuracy (%),Baseline Precision (%),Tiny Precision (%),Baseline Recall (%),Tiny Recall (%),...,Tiny Macro F1 (%),Baseline Infer. Time (ms/sample),Tiny Infer. Time (ms/sample),Baseline Model Size (MB),Tiny Model Size (MB),Weighted F1 Change (%),Macro F1 Change (%),Infer. Time Change (%),Model Size Change (%),Note
0,CICIDS2017,Weight Clustering-like Compression,DNN,DNN [Clustered],57.586999,57.745981,62.488000,62.519492,57.586999,57.745981,...,41.199515,0.075177,0.065290,15.795259,15.795259,0.104676,0.052709,-13.150871,0.0,Weight values were reassigned to learned share...
1,CICIDS2017,Weight Clustering-like Compression,VGG16 (1D-CNN),VGG16 (1D-CNN) [Clustered],53.011835,53.188483,61.291820,61.373475,53.011835,53.188483,...,39.337998,0.309818,0.333589,74.315339,74.315339,0.139096,0.122691,7.672690,0.0,Weight values were reassigned to learned share...
2,CICIDS2017,Weight Clustering-like Compression,VGG19 (1D-CNN),VGG19 (1D-CNN) [Clustered],64.423247,67.337926,75.045795,75.031707,64.423247,67.337926,...,58.537998,0.418013,0.362340,94.597700,94.597700,2.268331,1.453677,-13.318539,0.0,Weight values were reassigned to learned share...
3,CICIDS2017,Weight Clustering-like Compression,CNN,CNN [Clustered],80.003533,80.038862,64.107480,64.113123,80.003533,80.038862,...,44.456436,0.120391,0.095638,5.843764,5.843764,0.017459,0.010902,-20.560808,0.0,Weight values were reassigned to learned share...
4,CICIDS2017,Weight Clustering-like Compression,LSTM,LSTM [Clustered],80.021198,80.021198,64.110302,64.110302,80.021198,80.021198,...,44.450986,0.055777,0.056223,0.375109,0.375109,0.000000,0.000000,0.801220,0.0,Weight values were reassigned to learned share...


In [23]:

# Weight Clustering-like Compression — NF-ToN-IoT-V2

cluster_rows_nftoniotv2 = []

for model_name, model_obj in [
    ('DNN', dnn_nfton_model),
    ('VGG16 (1D-CNN)', vgg16_nfton_model),
    ('VGG19 (1D-CNN)', vgg19_nfton_model),
    ('CNN', cnn_nfton_model),
    ('LSTM', lstm_nfton_model),
    
]:
    X_test_ready = prepare_input_for_model(X_test_nfton, model_name)

    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()

    clustered_model = apply_weight_clustering_like(model_obj, n_clusters=16)
    save_path = save_tinyml_keras_model(clustered_model, 'NF-ToN-IoT-V2', model_name, 'Weight Clustering-like Compression')
    clustered_size_mb = get_keras_model_size_mb(clustered_model) * 1024 * 1024
    tiny_metrics = evaluate_keras_model(clustered_model, X_test_ready, y_test_nfton, f'{model_name} [Clustered]', model_size_bytes=clustered_size_mb)
    tiny_metrics['Saved Path'] = save_path

    cluster_rows_nftoniotv2.append(
        make_result_row(
            'NF-ToN-IoT-V2',
            'Weight Clustering-like Compression',
            model_name,
            f'{model_name} [Clustered]',
            baseline_metrics,
            tiny_metrics,
            note='Weight values were reassigned to learned shared cluster centers.'
        )
    )

cluster_df_nftoniotv2 = pd.DataFrame(cluster_rows_nftoniotv2)
display(cluster_df_nftoniotv2)

,Dataset,Technique,Baseline Model,Tiny Model,Baseline Accuracy (%),Tiny Accuracy (%),Baseline Precision (%),Tiny Precision (%),Baseline Recall (%),Tiny Recall (%),...,Tiny Macro F1 (%),Baseline Infer. Time (ms/sample),Tiny Infer. Time (ms/sample),Baseline Model Size (MB),Tiny Model Size (MB),Weighted F1 Change (%),Macro F1 Change (%),Infer. Time Change (%),Model Size Change (%),Note
0,NF-ToN-IoT-V2,Weight Clustering-like Compression,DNN,DNN [Clustered],97.450,97.300,97.454858,97.291475,97.450,97.300,...,96.593145,0.077564,0.063937,16.053071,16.053071,-0.158366,-0.209363,-17.569612,0.0,Weight values were reassigned to learned share...
1,NF-ToN-IoT-V2,Weight Clustering-like Compression,VGG16 (1D-CNN),VGG16 (1D-CNN) [Clustered],96.025,95.600,96.152499,95.770193,96.025,95.600,...,94.206026,0.298573,0.286554,74.317944,74.317944,-0.449281,-0.591414,-4.025415,0.0,Weight values were reassigned to learned share...
2,NF-ToN-IoT-V2,Weight Clustering-like Compression,VGG19 (1D-CNN),VGG19 (1D-CNN) [Clustered],72.575,72.575,52.671306,52.671306,72.575,72.575,...,42.054179,0.380053,0.373403,94.597709,94.597709,0.000000,0.000000,-1.749828,0.0,Weight values were reassigned to learned share...
3,NF-ToN-IoT-V2,Weight Clustering-like Compression,CNN,CNN [Clustered],97.575,97.475,97.578904,97.477330,97.575,97.475,...,96.781274,0.105730,0.108341,5.843784,5.843784,-0.100755,-0.127474,2.469759,0.0,Weight values were reassigned to learned share...
4,NF-ToN-IoT-V2,Weight Clustering-like Compression,LSTM,LSTM [Clustered],95.550,95.525,95.689190,95.610946,95.550,95.525,...,94.154162,0.059130,0.058927,0.375109,0.375109,-0.012069,-0.001093,-0.342529,0.0,Weight values were reassigned to learned share...


## Low-Rank Approximation

In [24]:

# Low-Rank Approximation — CICEVSE2024

low_rank_rows_cicevse2024 = []

for model_name, model_obj in [
    ('DNN', dnn_cicevse_model),
    ('VGG16 (1D-CNN)', vgg16_cicevse_model),
    ('VGG19 (1D-CNN)', vgg19_cicevse_model),
    ('CNN', cnn_cicevse_model),
    ('LSTM', lstm_cicevse_model),
    
]:
    X_test_ready = prepare_input_for_model(X_test_cicevse, model_name)

    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()

    low_rank_model = apply_low_rank_approximation(model_obj, rank_ratio=0.50)
    save_path = save_tinyml_keras_model(low_rank_model, 'CICEVSE2024', model_name, 'Low-Rank Approximation')
    low_rank_size_mb = get_keras_model_size_mb(low_rank_model) * 1024 * 1024
    tiny_metrics = evaluate_keras_model(low_rank_model, X_test_ready, y_test_cicevse, f'{model_name} [Low-Rank]', model_size_bytes=low_rank_size_mb)
    tiny_metrics['Saved Path'] = save_path

    low_rank_rows_cicevse2024.append(
        make_result_row(
            'CICEVSE2024',
            'Low-Rank Approximation',
            model_name,
            f'{model_name} [Low-Rank]',
            baseline_metrics,
            tiny_metrics,
            note='Dense and convolution-like weights were approximated with truncated SVD.'
        )
    )

low_rank_df_cicevse2024 = pd.DataFrame(low_rank_rows_cicevse2024)
display(low_rank_df_cicevse2024)

,Dataset,Technique,Baseline Model,Tiny Model,Baseline Accuracy (%),Tiny Accuracy (%),Baseline Precision (%),Tiny Precision (%),Baseline Recall (%),Tiny Recall (%),...,Tiny Macro F1 (%),Baseline Infer. Time (ms/sample),Tiny Infer. Time (ms/sample),Baseline Model Size (MB),Tiny Model Size (MB),Weighted F1 Change (%),Macro F1 Change (%),Infer. Time Change (%),Model Size Change (%),Note
0,CICEVSE2024,Low-Rank Approximation,DNN,DNN [Low-Rank],51.397969,43.928224,71.285165,43.654701,51.397969,43.928224,...,26.692361,0.064305,0.056380,16.583460,16.583460,-6.601029,-18.959472,-12.325289,0.0,Dense and convolution-like weights were approx...
1,CICEVSE2024,Low-Rank Approximation,VGG16 (1D-CNN),VGG16 (1D-CNN) [Low-Rank],59.465851,31.798581,65.214597,41.234258,59.465851,31.798581,...,19.782295,0.406534,0.368006,74.435253,74.435253,-30.043281,-34.406998,-9.477114,0.0,Dense and convolution-like weights were approx...
2,CICEVSE2024,Low-Rank Approximation,VGG19 (1D-CNN),VGG19 (1D-CNN) [Low-Rank],21.407706,21.407706,4.582899,4.582899,21.407706,21.407706,...,2.938818,0.522686,0.439462,94.715025,94.715025,0.000000,0.000000,-15.922356,0.0,Dense and convolution-like weights were approx...
3,CICEVSE2024,Low-Rank Approximation,CNN,CNN [Low-Rank],57.087217,21.171234,57.219110,20.518200,57.087217,21.171234,...,12.788870,0.163328,0.142724,5.858548,5.858548,-36.974217,-40.469468,-12.614957,0.0,Dense and convolution-like weights were approx...
4,CICEVSE2024,Low-Rank Approximation,LSTM,LSTM [Low-Rank],60.509111,26.860481,58.647871,27.664092,60.509111,26.860481,...,21.701733,0.065363,0.058008,0.382554,0.382554,-34.534187,-32.820423,-11.252227,0.0,Dense and convolution-like weights were approx...


In [25]:

# Low-Rank Approximation — CICIDS2017

low_rank_rows_cicids2017 = []

for model_name, model_obj in [
    ('DNN', dnn_cicids_model),
    ('VGG16 (1D-CNN)', vgg16_cicids_model),
    ('VGG19 (1D-CNN)', vgg19_cicids_model),
    ('CNN', cnn_cicids_model),
    ('LSTM', lstm_cicids_model),
    
]:
    X_test_ready = prepare_input_for_model(X_test_cicids, model_name)

    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()

    low_rank_model = apply_low_rank_approximation(model_obj, rank_ratio=0.50)
    save_path = save_tinyml_keras_model(low_rank_model, 'CICIDS2017', model_name, 'Low-Rank Approximation')
    low_rank_size_mb = get_keras_model_size_mb(low_rank_model) * 1024 * 1024
    tiny_metrics = evaluate_keras_model(low_rank_model, X_test_ready, y_test_cicids, f'{model_name} [Low-Rank]', model_size_bytes=low_rank_size_mb)
    tiny_metrics['Saved Path'] = save_path

    low_rank_rows_cicids2017.append(
        make_result_row(
            'CICIDS2017',
            'Low-Rank Approximation',
            model_name,
            f'{model_name} [Low-Rank]',
            baseline_metrics,
            tiny_metrics,
            note='Dense and convolution-like weights were approximated with truncated SVD.'
        )
    )

low_rank_df_cicids2017 = pd.DataFrame(low_rank_rows_cicids2017)
display(low_rank_df_cicids2017)

,Dataset,Technique,Baseline Model,Tiny Model,Baseline Accuracy (%),Tiny Accuracy (%),Baseline Precision (%),Tiny Precision (%),Baseline Recall (%),Tiny Recall (%),...,Tiny Macro F1 (%),Baseline Infer. Time (ms/sample),Tiny Infer. Time (ms/sample),Baseline Model Size (MB),Tiny Model Size (MB),Weighted F1 Change (%),Macro F1 Change (%),Infer. Time Change (%),Model Size Change (%),Note
0,CICIDS2017,Low-Rank Approximation,DNN,DNN [Low-Rank],57.586999,66.437025,62.488000,63.013997,57.586999,66.437025,...,42.035435,0.075177,0.060638,15.795259,15.795259,4.800249,0.888629,-19.339975,0.0,Dense and convolution-like weights were approx...
1,CICIDS2017,Low-Rank Approximation,VGG16 (1D-CNN),VGG16 (1D-CNN) [Low-Rank],53.011835,19.925808,61.291820,3.970378,53.011835,19.925808,...,16.615113,0.309818,0.255445,74.315339,74.315339,-50.012225,-22.600195,-17.549953,0.0,Dense and convolution-like weights were approx...
2,CICIDS2017,Low-Rank Approximation,VGG19 (1D-CNN),VGG19 (1D-CNN) [Low-Rank],64.423247,79.544250,75.045795,66.567174,64.423247,79.544250,...,44.725896,0.418013,0.393850,94.597700,94.597700,3.353993,-12.358425,-5.780559,0.0,Dense and convolution-like weights were approx...
3,CICIDS2017,Low-Rank Approximation,CNN,CNN [Low-Rank],80.003533,19.925808,64.107480,3.970378,80.003533,19.925808,...,16.615113,0.120391,0.105431,5.843764,5.843764,-64.557415,-27.830422,-12.426068,0.0,Dense and convolution-like weights were approx...
4,CICIDS2017,Low-Rank Approximation,LSTM,LSTM [Low-Rank],80.021198,80.074192,64.110302,64.118762,80.021198,80.074192,...,44.467334,0.055777,0.059619,0.375109,0.375109,0.026180,0.016348,6.889316,0.0,Dense and convolution-like weights were approx...


In [26]:

# Low-Rank Approximation — NF-ToN-IoT-V2

low_rank_rows_nftoniotv2 = []

for model_name, model_obj in [
    ('DNN', dnn_nfton_model),
    ('VGG16 (1D-CNN)', vgg16_nfton_model),
    ('VGG19 (1D-CNN)', vgg19_nfton_model),
    ('CNN', cnn_nfton_model),
    ('LSTM', lstm_nfton_model),
    
]:
    X_test_ready = prepare_input_for_model(X_test_nfton, model_name)

    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()

    low_rank_model = apply_low_rank_approximation(model_obj, rank_ratio=0.50)
    save_path = save_tinyml_keras_model(low_rank_model, 'NF-ToN-IoT-V2', model_name, 'Low-Rank Approximation')
    low_rank_size_mb = get_keras_model_size_mb(low_rank_model) * 1024 * 1024
    tiny_metrics = evaluate_keras_model(low_rank_model, X_test_ready, y_test_nfton, f'{model_name} [Low-Rank]', model_size_bytes=low_rank_size_mb)
    tiny_metrics['Saved Path'] = save_path

    low_rank_rows_nftoniotv2.append(
        make_result_row(
            'NF-ToN-IoT-V2',
            'Low-Rank Approximation',
            model_name,
            f'{model_name} [Low-Rank]',
            baseline_metrics,
            tiny_metrics,
            note='Dense and convolution-like weights were approximated with truncated SVD.'
        )
    )

low_rank_df_nftoniotv2 = pd.DataFrame(low_rank_rows_nftoniotv2)
display(low_rank_df_nftoniotv2)

,Dataset,Technique,Baseline Model,Tiny Model,Baseline Accuracy (%),Tiny Accuracy (%),Baseline Precision (%),Tiny Precision (%),Baseline Recall (%),Tiny Recall (%),...,Tiny Macro F1 (%),Baseline Infer. Time (ms/sample),Tiny Infer. Time (ms/sample),Baseline Model Size (MB),Tiny Model Size (MB),Weighted F1 Change (%),Macro F1 Change (%),Infer. Time Change (%),Model Size Change (%),Note
0,NF-ToN-IoT-V2,Low-Rank Approximation,DNN,DNN [Low-Rank],97.450,95.300,97.454858,95.275398,97.450,95.300,...,93.974660,0.077564,0.064661,16.053071,16.053071,-2.201611,-2.827848,-16.635711,0.0,Dense and convolution-like weights were approx...
1,NF-ToN-IoT-V2,Low-Rank Approximation,VGG16 (1D-CNN),VGG16 (1D-CNN) [Low-Rank],96.025,85.000,96.152499,84.982796,96.025,85.000,...,78.578888,0.298573,0.305346,74.317944,74.317944,-12.064339,-16.218552,2.268575,0.0,Dense and convolution-like weights were approx...
2,NF-ToN-IoT-V2,Low-Rank Approximation,VGG19 (1D-CNN),VGG19 (1D-CNN) [Low-Rank],72.575,72.575,52.671306,52.671306,72.575,72.575,...,42.054179,0.380053,0.414736,94.597709,94.597709,0.000000,0.000000,9.125834,0.0,Dense and convolution-like weights were approx...
3,NF-ToN-IoT-V2,Low-Rank Approximation,CNN,CNN [Low-Rank],97.575,72.575,97.578904,52.671306,97.575,72.575,...,42.054179,0.105730,0.113047,5.843784,5.843784,-36.515061,-54.854569,6.920062,0.0,Dense and convolution-like weights were approx...
4,NF-ToN-IoT-V2,Low-Rank Approximation,LSTM,LSTM [Low-Rank],95.550,64.300,95.689190,67.580286,95.550,64.300,...,58.637470,0.059130,0.056902,0.375109,0.375109,-29.897072,-35.517786,-3.767620,0.0,Dense and convolution-like weights were approx...


In [27]:
for model_name, model_obj in [
    ('DNN', dnn_cicevse_model),
    ('VGG16 (1D-CNN)', vgg16_cicevse_model),
    ('VGG19 (1D-CNN)', vgg19_cicevse_model),
    ('CNN', cnn_cicevse_model),
    ('LSTM', lstm_cicevse_model),
]:
    X_test_ready = prepare_input_for_model(X_test_cicevse, model_name)
    print(model_name)
    print("model input shape:", model_obj.input_shape)
    print("prepared shape:", X_test_ready.shape)
    print("-" * 40)

DNN
model input shape: (None, 85)
prepared shape: (7189, 85)
----------------------------------------
VGG16 (1D-CNN)
model input shape: (None, 85, 1)
prepared shape: (7189, 85, 1)
----------------------------------------
VGG19 (1D-CNN)
model input shape: (None, 85, 1)
prepared shape: (7189, 85, 1)
----------------------------------------
CNN
model input shape: (None, 85, 1)
prepared shape: (7189, 85, 1)
----------------------------------------
LSTM
model input shape: (None, 17, 5)
prepared shape: (7189, 17, 5)
----------------------------------------


## Dynamic Quantization

In [28]:

# Dynamic Quantization — CICEVSE2024

dynamic_quant_rows_cicevse2024 = []

for model_name, model_obj in [
    ('DNN', dnn_cicevse_model),
    ('VGG16 (1D-CNN)', vgg16_cicevse_model),
    ('VGG19 (1D-CNN)', vgg19_cicevse_model),
    ('CNN', cnn_cicevse_model),
    #('LSTM', lstm_cicevse_model),
    
]:
    X_test_ready = prepare_input_for_model(X_test_cicevse, model_name)

    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()
    tflite_path = get_tinyml_save_path('CICEVSE2024', model_name, 'Dynamic Quantization', 'tflite')
    export_tflite_model(model_obj, tflite_path, quantization='dynamic')

    tiny_metrics = evaluate_tflite_model(tflite_path, X_test_ready, y_test_cicevse, f'{model_name} [Dynamic TFLite]')
    tiny_metrics['Saved Path'] = tflite_path

    dynamic_quant_rows_cicevse2024.append(
        make_result_row(
            'CICEVSE2024',
            'Dynamic Quantization',
            model_name,
            f'{model_name} [Dynamic TFLite]',
            baseline_metrics,
            tiny_metrics,
            #note='Dynamic TFLite conversion was applied after loading the saved baseline model.'
        )
    )

dynamic_quant_df_cicevse2024 = pd.DataFrame(dynamic_quant_rows_cicevse2024)
display(dynamic_quant_df_cicevse2024)


INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmp5lom4xh8\assets


INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmp5lom4xh8\assets


Saved artifact at 'C:\Users\100987~1\AppData\Local\Temp\tmp5lom4xh8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 85), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 12), dtype=tf.float32, name=None)
Captures:
  2345388894992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345388896528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345388895760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345388896144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345388895184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345388895952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345388897104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345388898064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345388896336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345388897296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  23453888969

c:\Users\100987869\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmpgcebbl85\assets


...

INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmpyn45tuq5\assets


INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmpyn45tuq5\assets


Saved artifact at 'C:\Users\100987~1\AppData\Local\Temp\tmpyn45tuq5'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 85, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 12), dtype=tf.float32, name=None)
Captures:
  2345451547024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345451547984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345451547408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345451539344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345451547792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345451547600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345451548368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345451549328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345451548944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2345451548752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  23454515

c:\Users\100987869\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,Dataset,Technique,Baseline Model,Tiny Model,Baseline Accuracy (%),Tiny Accuracy (%),Baseline Precision (%),Tiny Precision (%),Baseline Recall (%),Tiny Recall (%),...,Tiny Macro F1 (%),Baseline Infer. Time (ms/sample),Tiny Infer. Time (ms/sample),Baseline Model Size (MB),Tiny Model Size (MB),Weighted F1 Change (%),Macro F1 Change (%),Infer. Time Change (%),Model Size Change (%),Note
0,CICEVSE2024,Dynamic Quantization,DNN,DNN [Dynamic TFLite],51.397969,47.336208,71.285165,66.868574,51.397969,47.336208,...,40.726533,0.064305,0.018248,16.583460,1.411827,-1.255053,-4.925300,-71.622583,-91.486535,
1,CICEVSE2024,Dynamic Quantization,VGG16 (1D-CNN),VGG16 (1D-CNN) [Dynamic TFLite],59.465851,59.312839,65.214597,65.634435,59.465851,59.312839,...,53.871135,0.406534,0.231684,74.435253,6.303802,-0.102984,-0.318159,-43.009874,-91.531160,
2,CICEVSE2024,Dynamic Quantization,VGG19 (1D-CNN),VGG19 (1D-CNN) [Dynamic TFLite],21.407706,21.407706,4.582899,4.582899,21.407706,21.407706,...,2.938818,0.522686,0.342485,94.715025,8.014137,0.000000,0.000000,-34.475897,-91.538684,
3,CICEVSE2024,Dynamic Quantization,CNN,CNN [Dynamic TFLite],57.087217,51.384059,57.219110,50.435755,57.087217,51.384059,...,33.252060,0.163328,0.066332,5.858548,0.519608,-5.049836,-20.006278,-59.386955,-91.130780,


In [ ]:
# Dynamic Quantization — CICIDS2017

dynamic_quant_rows_cicids2017 = []

for model_name, model_obj in [
    ('DNN',            dnn_cicids_model),
    ('VGG16 (1D-CNN)', vgg16_cicids_model),
    ('VGG19 (1D-CNN)', vgg19_cicids_model),
    ('CNN',            cnn_cicids_model),
    # LSTM excluded: TFLite dynamic quantization of stateful LSTM graphs is
    # unreliable; skip for consistency with CICEVSE2024 and NF-ToN blocks.
]:
    X_test_ready = prepare_input_for_model(X_test_cicids, model_name)

    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()
    tflite_path = get_tinyml_save_path('CICEVSE2024', model_name, 'Dynamic Quantization', 'tflite')
    export_tflite_model(model_obj, tflite_path, quantization='dynamic')

    tiny_metrics = evaluate_tflite_model(tflite_path, X_test_ready, y_test_cicids,
                                         f'{model_name} [Dynamic TFLite]')
    tiny_metrics['Saved Path'] = tflite_path

    dynamic_quant_rows_cicids2017.append(
        make_result_row(
            'CICIDS2017',
            'Dynamic Quantization',
            model_name,
            f'{model_name} [Dynamic TFLite]',
            baseline_metrics,
            tiny_metrics,
            note='Dynamic TFLite conversion was applied after loading the saved baseline model.'
        )
    )

dynamic_quant_df_cicids2017 = pd.DataFrame(dynamic_quant_rows_cicids2017)
display(dynamic_quant_df_cicids2017)


SyntaxError: invalid syntax. Perhaps you forgot a comma? (1880983597.py, line 17)

In [ ]:

# Dynamic Quantization — NF-ToN-IoT-V2

dynamic_quant_rows_nftoniotv2 = []

for model_name, model_obj in [
    ('DNN', dnn_nfton_model),
    ('VGG16 (1D-CNN)', vgg16_nfton_model),
    ('VGG19 (1D-CNN)', vgg19_nfton_model),
    ('CNN', cnn_nfton_model),
    #('LSTM', lstm_nfton_model)
]:
    X_test_ready = prepare_input_for_model(X_test_nfton, model_name)

    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()
    tflite_path = get_tinyml_save_path('NF-ToN-IoT-V2', model_name, 'Dynamic Quantization', 'tflite')
    export_tflite_model(model_obj, tflite_path, quantization='dynamic')

    tiny_metrics = evaluate_tflite_model(tflite_path, X_test_ready, y_test_nfton, f'{model_name} [Dynamic TFLite]')
    tiny_metrics['Saved Path'] = tflite_path

    dynamic_quant_rows_nftoniotv2.append(
        make_result_row(
            'NF-ToN-IoT-V2',
            'Dynamic Quantization',
            model_name,
            f'{model_name} [Dynamic TFLite]',
            baseline_metrics,
            tiny_metrics,
            note='Dynamic TFLite conversion was applied after loading the saved baseline model.'
        )
    )

dynamic_quant_df_nftoniotv2 = pd.DataFrame(dynamic_quant_rows_nftoniotv2)
display(dynamic_quant_df_nftoniotv2)


INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmpo46lja41\assets


INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmpo46lja41\assets


Saved artifact at 'C:\Users\100987~1\AppData\Local\Temp\tmpo46lja41'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 41), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  2029237219600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2029237217680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2029237218064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2029237217488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2029237217104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2029237218448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2029237215568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2029237221136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2029237221520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2029237213264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  202923721672

c:\Users\100987869\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmpcgk4wfxb\assets


...

INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmp7syd32to\assets


INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmp7syd32to\assets


Saved artifact at 'C:\Users\100987~1\AppData\Local\Temp\tmp7syd32to'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 41, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  2032271929808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032271930768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032271930192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032271922128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032271930576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032271930384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032271931152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032271932112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032271931728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032271931536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  203227193

c:\Users\100987869\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,Dataset,Technique,Baseline Model,Tiny Model,Baseline Accuracy (%),Tiny Accuracy (%),Baseline Precision (%),Tiny Precision (%),Baseline Recall (%),Tiny Recall (%),...,Tiny Macro F1 (%),Baseline Infer. Time (ms/sample),Tiny Infer. Time (ms/sample),Baseline Model Size (MB),Tiny Model Size (MB),Weighted F1 Change (%),Macro F1 Change (%),Infer. Time Change (%),Model Size Change (%),Note
0,NF-ToN-IoT-V2,Dynamic Quantization,DNN,DNN [Dynamic TFLite],97.450,97.650,97.454858,97.646529,97.450,97.650,...,97.043248,0.062815,0.019888,16.053071,1.368164,0.195833,0.240740,-68.338584,-91.477244,Dynamic TFLite conversion was applied after lo...
1,NF-ToN-IoT-V2,Dynamic Quantization,VGG16 (1D-CNN),VGG16 (1D-CNN) [Dynamic TFLite],96.025,96.025,96.152499,96.152499,96.025,96.025,...,94.797440,0.298258,0.163109,74.317944,6.293884,0.000000,0.000000,-45.312730,-91.531138,Dynamic TFLite conversion was applied after lo...
2,NF-ToN-IoT-V2,Dynamic Quantization,VGG19 (1D-CNN),VGG19 (1D-CNN) [Dynamic TFLite],72.575,72.575,52.671306,52.671306,72.575,72.575,...,42.054179,0.381671,0.285835,94.597709,8.004204,0.000000,0.000000,-25.109532,-91.538692,Dynamic TFLite conversion was applied after lo...
3,NF-ToN-IoT-V2,Dynamic Quantization,CNN,CNN [Dynamic TFLite],97.575,97.450,97.578904,97.482572,97.575,97.450,...,96.725016,0.102044,0.029255,5.843784,0.518906,-0.135979,-0.183733,-71.330767,-91.120383,Dynamic TFLite conversion was applied after lo...


## INT8 Quantization

In [ ]:

# INT8 Quantization — CICEVSE2024

int8_quant_rows_cicevse2024 = []

for model_name, model_obj in [
    ('DNN', dnn_cicevse_model),
    ('VGG16 (1D-CNN)', vgg16_cicevse_model),
    ('VGG19 (1D-CNN)', vgg19_cicevse_model),
    ('CNN', cnn_cicevse_model),
    #('LSTM', lstm_cicevse_model)
    
]:
    X_train_ready = prepare_input_for_model(X_train_cicevse, model_name)
    X_test_ready = prepare_input_for_model(X_test_cicevse, model_name)

    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()
    tflite_path = get_tinyml_save_path('CICEVSE2024', model_name, 'INT8 Quantization', 'tflite')
    export_tflite_model(
        model_obj,
        tflite_path,
        quantization='int8',
        representative_data=lambda X=X_train_ready: representative_dataset_generator(X)
    )

    tiny_metrics = evaluate_tflite_model(tflite_path, X_test_ready, y_test_cicevse, f'{model_name} [INT8 TFLite]')
    tiny_metrics['Saved Path'] = tflite_path

    int8_quant_rows_cicevse2024.append(
        make_result_row(
            'CICEVSE2024',
            'INT8 Quantization',
            model_name,
            f'{model_name} [INT8 TFLite]',
            baseline_metrics,
            tiny_metrics,
            note='INT8 TFLite conversion used representative calibration data from the corresponding training split.'
        )
    )

int8_quant_df_cicevse2024 = pd.DataFrame(int8_quant_rows_cicevse2024)
display(int8_quant_df_cicevse2024)


INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmpblztprk8\assets


INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmpblztprk8\assets


Saved artifact at 'C:\Users\100987~1\AppData\Local\Temp\tmpblztprk8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 85), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 12), dtype=tf.float32, name=None)
Captures:
  2032160624464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032160626960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032160624848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032160627920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032160627344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032160627728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032160625808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032160623888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032160624656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2032160625040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  20321606261

c:\Users\100987869\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


NameError: name 'allow_select_tf_ops' is not defined

In [ ]:

# INT8 Quantization — CICIDS2017

int8_quant_rows_cicids2017 = []

for model_name, model_obj in [
    ('DNN', dnn_cicids_model),
    ('VGG16 (1D-CNN)', vgg16_cicids_model),
    ('VGG19 (1D-CNN)', vgg19_cicids_model),
    ('CNN', cnn_cicids_model),
    ('LSTM', lstm_cicids_model)
]:
    X_train_ready = prepare_input_for_model(X_train_cicids, model_name)
    X_test_ready = prepare_input_for_model(X_test_cicids, model_name)

    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()
    tflite_path = get_tinyml_save_path('CICIDS2017', model_name, 'INT8 Quantization', 'tflite')
    export_tflite_model(
        model_obj,
        tflite_path,
        quantization='int8',
        representative_data=lambda X=X_train_ready: representative_dataset_generator(X)
    )

    tiny_metrics = evaluate_tflite_model(tflite_path, X_test_ready, y_test_cicids, f'{model_name} [INT8 TFLite]')
    tiny_metrics['Saved Path'] = tflite_path

    int8_quant_rows_cicids2017.append(
        make_result_row(
            'CICIDS2017',
            'INT8 Quantization',
            model_name,
            f'{model_name} [INT8 TFLite]',
            baseline_metrics,
            tiny_metrics,
            note='INT8 TFLite conversion used representative calibration data from the corresponding training split.'
        )
    )

int8_quant_df_cicids2017 = pd.DataFrame(int8_quant_rows_cicids2017)
display(int8_quant_df_cicids2017)


In [ ]:

# INT8 Quantization — NF-ToN-IoT-V2

int8_quant_rows_nftoniotv2 = []

for model_name, model_obj in [
    ('DNN', dnn_nfton_model),
    ('VGG16 (1D-CNN)', vgg16_nfton_model),
    ('VGG19 (1D-CNN)', vgg19_nfton_model),
    ('CNN', cnn_nfton_model),
    ('LSTM', lstm_nfton_model)
    
]:
    X_train_ready = prepare_input_for_model(X_train_nfton, model_name)
    X_test_ready = prepare_input_for_model(X_test_nfton, model_name)

    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()
    tflite_path = get_tinyml_save_path('NF-ToN-IoT-V2', model_name, 'INT8 Quantization', 'tflite')
    export_tflite_model(
        model_obj,
        tflite_path,
        quantization='int8',
        representative_data=lambda X=X_train_ready: representative_dataset_generator(X)
    )

    tiny_metrics = evaluate_tflite_model(tflite_path, X_test_ready, y_test_nfton, f'{model_name} [INT8 TFLite]')
    tiny_metrics['Saved Path'] = tflite_path

    int8_quant_rows_nftoniotv2.append(
        make_result_row(
            'NF-ToN-IoT-V2',
            'INT8 Quantization',
            model_name,
            f'{model_name} [INT8 TFLite]',
            baseline_metrics,
            tiny_metrics,
            note='INT8 TFLite conversion used representative calibration data from the corresponding training split.'
        )
    )

int8_quant_df_nftoniotv2 = pd.DataFrame(int8_quant_rows_nftoniotv2)
display(int8_quant_df_nftoniotv2)


INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmp3u_aq9px\assets


INFO:tensorflow:Assets written to: C:\Users\100987~1\AppData\Local\Temp\tmp3u_aq9px\assets


Saved artifact at 'C:\Users\100987~1\AppData\Local\Temp\tmp3u_aq9px'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 41), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  2290206274960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2290206274384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2290206274192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2290206269200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2290206266128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2290206274000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2290206272464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2290206267856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2290206269392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2290206264784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  229020627323

c:\Users\100987869\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
c:\Users\100987869\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,Dataset,Technique,Baseline Model,Tiny Model,Baseline Accuracy (%),Tiny Accuracy (%),Baseline Precision (%),Tiny Precision (%),Baseline Recall (%),Tiny Recall (%),...,Tiny Macro F1 (%),Baseline Infer. Time (ms/sample),Tiny Infer. Time (ms/sample),Baseline Model Size (MB),Tiny Model Size (MB),Weighted F1 Change (%),Macro F1 Change (%),Infer. Time Change (%),Model Size Change (%),Note
0,NF-ToN-IoT-V2,INT8 Quantization,DNN,DNN [INT8 TFLite],97.775,97.734,97.773152,97.733788,97.775,97.734,...,97.118002,0.073708,0.024586,16.053071,1.398964,-0.054177,-0.084209,-66.644214,-91.285381,INT8 TFLite conversion used representative cal...


## Combined Task 1 Result Table

In [ ]:

# TinyML saved-artifact audit
# Run this after all TinyML blocks complete to confirm outputs are organized correctly.

tinyml_saved_rows = []
for root, _, files in os.walk(TINYML_MODEL_ROOT):
    for file in files:
        if file.endswith(('.keras', '.tflite')):
            path = os.path.join(root, file)
            tinyml_saved_rows.append({
                'Path': path,
                'File': file,
                'Size (MB)': os.path.getsize(path) / (1024 * 1024),
            })

tinyml_saved_artifacts_df = pd.DataFrame(tinyml_saved_rows).sort_values('Path') if tinyml_saved_rows else pd.DataFrame(columns=['Path', 'File', 'Size (MB)'])
display(tinyml_saved_artifacts_df)
print(f'Total TinyML saved artifacts: {len(tinyml_saved_artifacts_df)}')
print(f'TinyML root folder: {os.path.abspath(TINYML_MODEL_ROOT)}')


In [ ]:
task1_deep_df = pd.concat([
    pruning_df_cicevse2024,
    pruning_df_cicids2017,
    pruning_df_nftoniotv2,
    cluster_df_cicevse2024,
    cluster_df_cicids2017,
    cluster_df_nftoniotv2,
    low_rank_df_cicevse2024,
    low_rank_df_cicids2017,
    low_rank_df_nftoniotv2,
    dynamic_quant_df_cicevse2024,
    dynamic_quant_df_cicids2017,
    dynamic_quant_df_nftoniotv2,
    int8_quant_df_cicevse2024,
    int8_quant_df_cicids2017,
    int8_quant_df_nftoniotv2,
], ignore_index=True)

display(task1_deep_df)


,Dataset,Technique,Baseline Model,Tiny Model,Baseline Accuracy (%),Tiny Accuracy (%),Baseline Precision (%),Tiny Precision (%),Baseline Recall (%),Tiny Recall (%),...,Tiny Macro F1 (%),Baseline Infer. Time (ms/sample),Tiny Infer. Time (ms/sample),Baseline Model Size (MB),Tiny Model Size (MB),Weighted F1 Change (%),Macro F1 Change (%),Infer. Time Change (%),Model Size Change (%),Note
0,CICEVSE2024,Pruning-like Sparsification,DNN,DNN [Pruned],50.813743,47.976075,69.391120,69.733887,50.813743,47.976075,...,39.880436,0.113603,0.095732,16.583460,16.583460,-0.295175,-3.246431,-15.731350,0.000000,Weights below the selected quantile threshold ...
1,CICEVSE2024,Weight Clustering-like Compression,DNN,DNN [Clustered],50.813743,51.982195,69.391120,63.292803,50.813743,51.982195,...,51.704899,0.113603,0.107860,16.583460,16.583460,1.677880,8.578033,-5.055507,0.000000,Weight values were reassigned to learned share...
2,NF-ToN-IoT-V2,Low-Rank Approximation,DNN,DNN [Low-Rank],97.775000,83.545000,97.773152,87.094989,97.775000,83.545000,...,81.404877,0.073708,0.071614,16.053071,16.053071,-13.519747,-15.797334,-2.841830,0.000000,Dense and convolution-like weights were approx...
3,CICEVSE2024,Dynamic Quantization,DNN,DNN [Dynamic TFLite],50.813743,47.836973,69.391120,73.509409,50.813743,47.836973,...,42.585461,0.113603,0.021935,16.583460,1.411827,-0.720057,-0.541406,-80.691868,-91.486535,Dynamic TFLite conversion was applied after lo...
4,CICEVSE2024,INT8 Quantization,DNN,DNN [INT8 TFLite],50.813743,48.504660,69.391120,70.811697,50.813743,48.504660,...,42.649917,0.113603,0.025801,16.583460,1.443420,-0.542758,-0.476950,-77.288026,-91.296024,INT8 TFLite conversion used representative cal...
5,NF-ToN-IoT-V2,INT8 Quantization,DNN,DNN [INT8 TFLite],97.775000,97.734000,97.773152,97.733788,97.775000,97.734000,...,97.118002,0.073708,0.024586,16.053071,1.398964,-0.054177,-0.084209,-66.644214,-91.285381,INT8 TFLite conversion used representative cal...


## Final Task 1 Plots

In [ ]:
# ── Comprehensive Task 1 Summary Plot ──────────────────────────────────────
# A single figure with 4 subplots (Weighted F1 change, Macro F1 change,
# Inference time change, Model size change), coloured by technique and shaped
# by dataset, plus a scatter of F1 vs inference time.

import matplotlib.gridspec as gridspec

TECHNIQUE_PALETTE = {
    'Pruning-like Sparsification':        '#4C72B0',
    'Weight Clustering-like Compression': '#DD8452',
    'Low-Rank Approximation':             '#55A868',
    'Dynamic Quantization':               '#C44E52',
    'INT8 Quantization':                  '#8172B2',
}
DATASET_MARKERS = {
    'CICEVSE2024':   'o',
    'CICIDS2017':    's',
    'NF-ToN-IoT-V2': '^',
}

plot_df = task1_deep_df.copy()

fig = plt.figure(figsize=(22, 26))
fig.suptitle(
    'Task 1 — Deep TinyML Techniques: Comprehensive Summary\n'
    '(All models × All datasets × All techniques)',
    fontsize=15, fontweight='bold', y=0.98
)

gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.42, wspace=0.32)

bar_specs = [
    (gs[0, 0], 'Weighted F1 Change (%)',       'Weighted F1 Δ (%)'),
    (gs[0, 1], 'Macro F1 Change (%)',           'Macro F1 Δ (%)'),
    (gs[1, 0], 'Infer. Time Change (%)',        'Inference Time Δ (%)'),
    (gs[1, 1], 'Model Size Change (%)',         'Model Size Δ (%)'),
]

for spec, col, xlabel in bar_specs:
    ax = fig.add_subplot(spec)
    sub = plot_df.sort_values(col, ascending=True).copy()
    colors = [TECHNIQUE_PALETTE.get(t, '#888888') for t in sub['Technique']]
    bars = ax.barh(sub['Tiny Model'], sub[col], color=colors, edgecolor='white', linewidth=0.4)
    ax.axvline(0, color='black', linewidth=0.7, linestyle='--')
    for bar, val in zip(bars, sub[col]):
        offset = 0.15 if val >= 0 else -0.15
        ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}', va='center', ha='left' if val >= 0 else 'right', fontsize=6)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=6)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.grid(True, axis='x', linestyle='--', alpha=0.4)

# ── Scatter: F1 vs Inference time, coloured by technique, shaped by dataset ──
ax_scatter = fig.add_subplot(gs[2, :])
for technique, tdf in plot_df.groupby('Technique'):
    for dataset, ddf in tdf.groupby('Dataset'):
        ax_scatter.scatter(
            ddf['Tiny Infer. Time (ms/sample)'],
            ddf['Tiny Weighted F1 (%)'],
            label=f'{technique} / {dataset}',
            color=TECHNIQUE_PALETTE.get(technique, '#888888'),
            marker=DATASET_MARKERS.get(dataset, 'D'),
            s=90, alpha=0.85, edgecolors='white', linewidths=0.5
        )
        for _, row in ddf.iterrows():
            ax_scatter.annotate(
                row['Baseline Model'],
                (row['Tiny Infer. Time (ms/sample)'], row['Tiny Weighted F1 (%)']),
                fontsize=5.5, ha='left', va='bottom',
                xytext=(3, 2), textcoords='offset points'
            )

ax_scatter.set_xlabel('Tiny Inference Time (ms / sample)', fontsize=10)
ax_scatter.set_ylabel('Tiny Weighted F1 (%)', fontsize=10)
ax_scatter.set_title('Accuracy–Efficiency Frontier: Weighted F1 vs Inference Time', fontsize=11, fontweight='bold')
ax_scatter.grid(True, linestyle='--', alpha=0.4)

# Legend — split into two columns to keep it compact
handles, labels = ax_scatter.get_legend_handles_labels()
ax_scatter.legend(handles, labels, fontsize=6.5, ncol=3,
                  loc='lower right', framealpha=0.85, title='Technique / Dataset', title_fontsize=7)

# Shared technique legend patch
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=t) for t, c in TECHNIQUE_PALETTE.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=5,
           fontsize=8, title='Technique (colour)', title_fontsize=9,
           bbox_to_anchor=(0.5, 0.01), framealpha=0.9)

plt.savefig('task1_tinyml_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Summary plot saved to task1_tinyml_summary.png")
